In [5]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# -----------------------------------------------------------------
# THE INNER GRAPH (the "department" with its own internal steps)
# -----------------------------------------------------------------
class InnerState(TypedDict):
    number: int

def inner_step_1(state: InnerState) -> dict:
    print("  [inner] step 1: adding 5")
    return {"number": state["number"] + 5}

def inner_step_2(state: InnerState) -> dict:
    print("  [inner] step 2: multiplying by 2")
    return {"number": state["number"] * 2}

inner_builder = StateGraph(InnerState)
inner_builder.add_node("inner_step_1", inner_step_1)
inner_builder.add_node("inner_step_2", inner_step_2)
inner_builder.add_edge(START, "inner_step_1")
inner_builder.add_edge("inner_step_1", "inner_step_2")
inner_builder.add_edge("inner_step_2", END)

inner_graph = inner_builder.compile()   # this is now a fully working graph on its own


In [9]:
# -----------------------------------------------------------------
# THE OUTER GRAPH (the "company") — plugs the inner graph in as a node
# -----------------------------------------------------------------
class OuterState(TypedDict):
    number: int

def outer_step_1(state: OuterState) -> dict:
    print("[outer] step 1: subtracting 1")
    return {"number": state["number"] - 1}

outer_builder = StateGraph(OuterState)
outer_builder.add_node("outer_step_1", outer_step_1)

# THIS is the key line: add the compiled inner_graph directly as a node.
# Since both states share the same field name "number", LangGraph can
# pass state between them automatically.
outer_builder.add_node("inner_graph", inner_graph)

outer_builder.add_edge(START, "outer_step_1")
outer_builder.add_edge("outer_step_1", "inner_graph")
outer_builder.add_edge("inner_graph", END)

outer_graph = outer_builder.compile()


if __name__ == "__main__":
    result = outer_graph.invoke({"number": 10})
    print("Final result:", result)
    # Trace: 10 -1 = 9 -> [inner: 9+5=14 -> 14*2=28]
    # Final should be {'number': 28}
outer_graph.get_graph().print_ascii()

[outer] step 1: subtracting 1
  [inner] step 1: adding 5
  [inner] step 2: multiplying by 2
Final result: {'number': 28}
  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
+--------------+ 
| outer_step_1 | 
+--------------+ 
        *        
        *        
        *        
+-------------+  
| inner_graph |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    
